# 03 -- Comparisons, Error Diagnostics & Operational Action Playbook

**Project:** FlyRank Ranking Signal Analysis -- Predicting Continuous Search Visibility  
**Pipeline Stage:** Notebook 03 of 03 (Model Evaluation, Error Diagnostics & Operational Triage)  
**Target Variable:** Continuous Search Visibility $\log(1 + \text{impressions\_90d})$ on Held-Out Enterprise Client Portfolios  
**Standardized Split:** Grouped Holdout (`GroupShuffleSplit`, `test_size=0.20`, `random_state=42`) with Zero Client Domain Overlap  

---

### Notebook Architecture & Objectives
1. **Benchmark Performance Comparisons:** Compile the official evaluation table comparing Baselines vs. Candidate Models.
2. **Diagnostic Error Audits:**
   - Visual inspection of Actual vs. Predicted scatter, residual distribution, and absolute error boxplots.
   - Sub-cohort performance analysis across content types and search intents.
   - In-depth post-mortem audit of 3 concrete outlier cases.
3. **Operational Action Playbook:**
   - Compute the **Opportunity Residual Gap** ($\hat{y} - y$) to detect under-indexed high-potential content.
   - Assign operational triage reason codes (`under_indexed_decay`, `high_demand_opportunity`, `stale_authority`).
   - Export the prioritized refresh queue to `work/outputs/refresh_queue_sample.csv`.

## 1. Official Benchmark Comparisons

All models are evaluated on the standardized, held-out test cohort of **6,163 content items across 7 unseen clients**:
* **Primary Metric:** $R^2$ Score (Coefficient of Determination) -- proportion of variance explained on unseen client domains.
* **Secondary Error Metrics:** RMSE and MAE (magnitude of prediction error in natural log units).
* **Monotonic Ranking Quality:** Spearman rank correlation ($\rho$) -- measure of sorting accuracy for prioritizing content queues.

In [1]:
# ============================================================
# Code Cell 1: Benchmark Table & Metrics JSON (Step 3.1)
# ============================================================
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, json, pathlib, os, warnings, sys
from scipy import stats
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance
warnings.filterwarnings('ignore')
if sys.stdout.encoding != 'utf-8':
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

curr = pathlib.Path.cwd().resolve()
while curr.name != 'DA-Project' and curr != curr.parent:
    curr = curr.parent
PROJECT_ROOT = curr
OUTPUT_DIR = PROJECT_ROOT / 'work' / 'outputs'

# Load payload
with open(OUTPUT_DIR / 'model_evaluation_payload.pkl', 'rb') as f:
    payload = pickle.load(f)

y_test = payload['y_test']
pred_dummy = payload['pred_dummy']
pred_ols = payload['pred_ols']
pred_ridge = payload['pred_ridge']
pred_hgbr = payload['pred_hgbr']
pred_champion = payload['pred_champion']
client_ids = payload['client_id']
content_ids = payload['content_id']

def evaluate_model(y_true, y_pred, name):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    if np.all(y_pred == y_pred[0]):
        spearman_r = 0.0
    else:
        spearman_r, _ = stats.spearmanr(y_true, y_pred)
        if np.isnan(spearman_r):
            spearman_r = 0.0
    return {'model': name, 'R2': round(float(r2), 4), 'RMSE': round(float(rmse), 4),
            'MAE': round(float(mae), 4), 'Spearman': round(float(spearman_r), 4)}

# Compute metrics for all models
rmse_baseline = np.sqrt(mean_squared_error(y_test, pred_dummy))

models = [
    ('Baseline 1: Mean Dummy', pred_dummy),
    ('Baseline 2: OLS log_word_count', pred_ols),
    ('Candidate A: RidgeCV + Interactions', pred_ridge),
    ('Candidate B: HGBR (tuned)', pred_hgbr),
    ('Champion: 0.5*Ridge + 0.5*HGBR', pred_champion),
]

all_metrics = []
for name, pred in models:
    m = evaluate_model(y_test, pred, name)
    rmse_reduction = (rmse_baseline - m['RMSE']) / rmse_baseline * 100
    m['RMSE_reduction_pct'] = round(float(rmse_reduction), 2)
    all_metrics.append(m)

bench_df = pd.DataFrame(all_metrics)
print("=" * 90)
print("OFFICIAL BENCHMARK PERFORMANCE TABLE")
print("=" * 90)
print(bench_df.to_string(index=False))

# Export to JSON
metrics_json = {m['model']: {k: v for k, v in m.items() if k != 'model'} for m in all_metrics}
metrics_json['Ablation Champion'] = payload.get('metrics_ablation', {})
metrics_json['Leakage Sanity - Random Split'] = payload.get('metrics_random_split', {})
metrics_json['Leakage Sanity - Grouped Split'] = payload.get('metrics_grouped_split', {})

with open(OUTPUT_DIR / 'model_benchmark_metrics.json', 'w') as f:
    json.dump(metrics_json, f, indent=2)
print(f"\n[PASS] Saved model_benchmark_metrics.json")

OFFICIAL BENCHMARK PERFORMANCE TABLE
                              model      R2   RMSE    MAE  Spearman  RMSE_reduction_pct
             Baseline 1: Mean Dummy -0.0476 2.7457 2.2757    0.0000               -0.00
     Baseline 2: OLS log_word_count -0.0471 2.7451 2.2859   -0.0571                0.02
Candidate A: RidgeCV + Interactions  0.4460 1.9966 1.6457    0.6763               27.28
          Candidate B: HGBR (tuned)  0.5401 1.8193 1.4628    0.7277               33.74
     Champion: 0.5*Ridge + 0.5*HGBR  0.5290 1.8411 1.5143    0.7265               32.95

[PASS] Saved model_benchmark_metrics.json


In [2]:
# ============================================================
# Code Cell 2: Diagnostic Error Visualizations (Step 3.2)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
fig.suptitle('Champion Model: Diagnostic Evaluation Plots', fontsize=14, fontweight='bold')

# Plot 1: Actual vs Predicted
ax = axes[0]
ax.scatter(y_test, pred_champion, alpha=0.15, s=8, c='steelblue', edgecolors='none')
lims = [min(y_test.min(), pred_champion.min()) - 0.5,
        max(y_test.max(), pred_champion.max()) + 0.5]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='y = y_hat')
ax.set_xlabel('Actual log(1+impressions)')
ax.set_ylabel('Predicted log(1+impressions)')
ax.set_title('Actual vs Predicted')
ax.legend()
ax.set_xlim(lims)
ax.set_ylim(lims)

# Plot 2: Residual histogram + KDE
ax = axes[1]
residuals = pred_champion - y_test
ax.hist(residuals, bins=60, density=True, alpha=0.6, color='steelblue', edgecolor='white')
from scipy.stats import gaussian_kde
kde = gaussian_kde(residuals)
x_kde = np.linspace(residuals.min(), residuals.max(), 200)
ax.plot(x_kde, kde(x_kde), 'r-', linewidth=2, label='KDE')
ax.axvline(0, color='black', linestyle='--', linewidth=1.5, label='Zero bias')
ax.set_xlabel('Residual (Predicted - Actual)')
ax.set_ylabel('Density')
ax.set_title('Residual Distribution')
ax.legend()

# Plot 3: Absolute error boxplot
ax = axes[2]
abs_errors = pd.DataFrame({
    'Baseline 1\n(Mean)': np.abs(pred_dummy - y_test),
    'Baseline 2\n(OLS wc)': np.abs(pred_ols - y_test),
    'Champion': np.abs(pred_champion - y_test),
})
abs_errors.boxplot(ax=ax, showfliers=False)
ax.set_ylabel('Absolute Error (log-impressions)')
ax.set_title('Absolute Error Comparison')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'model_evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f"[PASS] Saved model_evaluation_plots.png")

[PASS] Saved model_evaluation_plots.png


In [3]:
# ============================================================
# Code Cell 3: Sub-Cohort Audit & Outlier Case Studies (Steps 3.3 & 3.4)
# ============================================================

# Load original data for sub-cohort columns
RAW_CSV = PROJECT_ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
df_raw = pd.read_csv(RAW_CSV)

# Build test frame with predictions
test_frame = pd.DataFrame({
    'client_id': client_ids,
    'content_id': content_ids,
    'y_actual': y_test,
    'y_pred_champion': pred_champion,
    'y_pred_dummy': pred_dummy,
    'y_pred_ols': pred_ols,
})
test_frame['residual'] = test_frame['y_pred_champion'] - test_frame['y_actual']
test_frame['abs_error'] = np.abs(test_frame['residual'])

# Merge raw columns for sub-cohort analysis
test_frame = test_frame.merge(
    df_raw[['content_id', 'content_type', 'main_intent', 'impressions_90d',
            'word_count', 'search_volume', 'avg_position', 'content_age_days',
            'days_since_last_update', 'sessions_90d', 'engagement_rate']],
    on='content_id', how='left'
)

# --- 3.3 Sub-Cohort Audit by content_type ---
print("=" * 80)
print("SUB-COHORT AUDIT: by content_type")
print("=" * 80)
for ct in test_frame['content_type'].dropna().unique():
    sub = test_frame[test_frame['content_type'] == ct]
    n = len(sub)
    reliability = '' if n >= 50 else ' [UNRELIABLE: n < 50]'
    r2 = r2_score(sub['y_actual'], sub['y_pred_champion']) if n > 1 else float('nan')
    sp, _ = stats.spearmanr(sub['y_actual'], sub['y_pred_champion']) if n > 2 else (float('nan'), 0)
    print(f"  {ct:20s}: n={n:4d}, R2={r2:.4f}, Spearman={sp:.4f}{reliability}")

print(f"\n{'=' * 80}")
print("SUB-COHORT AUDIT: by main_intent")
print("=" * 80)
for mi in test_frame['main_intent'].dropna().unique():
    sub = test_frame[test_frame['main_intent'] == mi]
    n = len(sub)
    reliability = '' if n >= 50 else ' [UNRELIABLE: n < 50]'
    r2 = r2_score(sub['y_actual'], sub['y_pred_champion']) if n > 1 else float('nan')
    sp, _ = stats.spearmanr(sub['y_actual'], sub['y_pred_champion']) if n > 2 else (float('nan'), 0)
    print(f"  {mi:20s}: n={n:4d}, R2={r2:.4f}, Spearman={sp:.4f}{reliability}")

# --- 3.4 Three Outlier Case Studies ---
print(f"\n{'=' * 80}")
print("OUTLIER CASE STUDIES")
print("=" * 80)

# Case 1: Under-predicted high performer (actual >> predicted)
high_actual = test_frame[test_frame['y_actual'] > test_frame['y_actual'].quantile(0.90)]
case1 = high_actual.nsmallest(1, 'residual').iloc[0]
print(f"\n--- Case 1: Under-Predicted High Performer ---")
print(f"  content_id: {case1['content_id']}")
print(f"  Actual log-impressions: {case1['y_actual']:.3f} (raw: {case1['impressions_90d']:,.0f})")
print(f"  Predicted: {case1['y_pred_champion']:.3f}")
print(f"  Residual: {case1['residual']:.3f}")
print(f"  word_count: {case1['word_count']}, search_volume: {case1['search_volume']}")
print(f"  avg_position: {case1['avg_position']}, content_age_days: {case1['content_age_days']}")
print(f"  content_type: {case1['content_type']}, main_intent: {case1['main_intent']}")
print(f"  Diagnosis: Observed impressions far exceed model predictions based on structural")
print(f"  signals. This is consistent with exogenous tailwinds not captured by pre-ranking")
print(f"  metrics, such as an off-site viral backlink surge, domain-level brand navigation,")
print(f"  or inclusion in Google Discover / top stories.")

# Case 2: Over-predicted low performer (predicted >> actual)
low_actual = test_frame[test_frame['y_actual'] < test_frame['y_actual'].quantile(0.30)]
case2 = low_actual.nlargest(1, 'residual').iloc[0]
print(f"\n--- Case 2: Over-Predicted Low Performer ---")
print(f"  content_id: {case2['content_id']}")
print(f"  Actual log-impressions: {case2['y_actual']:.3f} (raw: {case2['impressions_90d']:,.0f})")
print(f"  Predicted: {case2['y_pred_champion']:.3f}")
print(f"  Residual: {case2['residual']:.3f}")
print(f"  word_count: {case2['word_count']}, search_volume: {case2['search_volume']}")
print(f"  avg_position: {case2['avg_position']}, content_age_days: {case2['content_age_days']}")
print(f"  content_type: {case2['content_type']}, main_intent: {case2['main_intent']}")
print(f"  Diagnosis: Despite favorable pre-ranking signals (e.g. substantial target search")
print(f"  volume or length), realized impressions remain depressed. This is consistent with")
print(f"  severe SERP feature displacement (AI Overviews, video carousels), high keyword")
print(f"  difficulty crowding, or intent misalignment.")

# Case 3: Prime refresh opportunity (pred > actual, high gap, high age)
refresh_candidates = test_frame[
    (test_frame['residual'] > 1.0) &
    (test_frame['content_age_days'] > 365)
].copy()
if len(refresh_candidates) > 0:
    case3 = refresh_candidates.nlargest(1, 'residual').iloc[0]
else:
    case3 = test_frame.nlargest(3, 'residual').iloc[-1]
print(f"\n--- Case 3: Prime Refresh Opportunity ---")
print(f"  content_id: {case3['content_id']}")
print(f"  Actual log-impressions: {case3['y_actual']:.3f} (raw: {case3.get('impressions_90d', 'N/A')})")
print(f"  Predicted: {case3['y_pred_champion']:.3f}")
print(f"  Residual (opportunity gap): {case3['residual']:.3f}")
print(f"  word_count: {case3['word_count']}, search_volume: {case3['search_volume']}")
print(f"  avg_position: {case3['avg_position']}, content_age_days: {case3['content_age_days']}")
print(f"  days_since_last_update: {case3['days_since_last_update']}")
print(f"  content_type: {case3['content_type']}, main_intent: {case3['main_intent']}")
print(f"  Diagnosis: This aged URL exhibits high structural potential and topical relevance,")
print(f"  yet realized visibility has declined over time. The substantial opportunity gap")
print(f"  indicates content decay on an established asset -- an optimal target for editorial")
print(f"  refresh and internal link reinforcement.")

SUB-COHORT AUDIT: by content_type
  keyword article     : n=6163, R2=0.5290, Spearman=0.7265

SUB-COHORT AUDIT: by main_intent
  transactional       : n=1152, R2=0.4852, Spearman=0.7035
  informational       : n=4000, R2=0.5519, Spearman=0.7395
  commercial          : n= 986, R2=0.4699, Spearman=0.7053
  navigational        : n=   7, R2=0.5303, Spearman=0.7143 [UNRELIABLE: n < 50]

OUTLIER CASE STUDIES

--- Case 1: Under-Predicted High Performer ---
  content_id: content_8ba781dafa55
  Actual log-impressions: 9.690 (raw: 16,156)
  Predicted: 3.378
  Residual: -6.312
  word_count: 3343.0, search_volume: 2900.0
  avg_position: 9.0, content_age_days: 348
  content_type: keyword article, main_intent: informational
  Diagnosis: Observed impressions far exceed model predictions based on structural
  signals. This is consistent with exogenous tailwinds not captured by pre-ranking
  metrics, such as an off-site viral backlink surge, domain-level brand navigation,
  or inclusion in Google Disco

In [4]:
# ============================================================
# Code Cell 4: Refresh Queue & Permutation Importance (Step 3.5)
# ============================================================

# --- 3.5 Build the Automated Content Refresh Queue ---
test_frame['opportunity_gap'] = test_frame['y_pred_champion'] - test_frame['y_actual']
test_frame['pred_impressions'] = np.expm1(test_frame['y_pred_champion'])
test_frame['actual_impressions'] = test_frame['impressions_90d']

# Filter for gap > 0.5
queue = test_frame[test_frame['opportunity_gap'] > 0.5].copy()
print(f"Pages with opportunity_gap > 0.5: {len(queue):,}")

# Assign reason codes with explicit documented thresholds
def assign_reason_code(row):
    reasons = []
    sv = row.get('search_volume', 0)
    er = row.get('engagement_rate', 0)
    cad = row.get('content_age_days', 0)
    dslu = row.get('days_since_last_update', 0)
    pos = row.get('avg_position', 0)
    
    # high_demand_opportunity: search_volume > 300 and engagement_rate > 20
    if (not pd.isna(sv) and sv > 300) and (not pd.isna(er) and er > 20):
        reasons.append('high_demand_opportunity')
    # stale_authority: content_age_days > 180 and days_since_last_update > 90
    if (not pd.isna(cad) and cad > 180) and (not pd.isna(dslu) and dslu > 90):
        reasons.append('stale_authority')
    # under_indexed_decay: high gap with trailing position (pos > 10 or unranked)
    if pos > 10 or pos == 0 or not reasons:
        reasons.append('under_indexed_decay')
    
    return '; '.join(reasons)

queue['reason_code'] = queue.apply(assign_reason_code, axis=1)

# Rank by opportunity gap (priority)
queue = queue.sort_values('opportunity_gap', ascending=False)

# Export top 50
top50 = queue.head(50)[[
    'client_id', 'content_id', 'pred_impressions', 'actual_impressions',
    'opportunity_gap', 'reason_code'
]].copy()

# Verify assertions
assert len(top50) == 50, f"Expected 50 rows, got {len(top50)}"
assert (top50['opportunity_gap'] > 0.5).all(), "Some rows have gap <= 0.5!"

top50.to_csv(OUTPUT_DIR / 'refresh_queue_sample.csv', index=False)
print(f"\n[PASS] Saved refresh_queue_sample.csv ({len(top50)} rows)")
print(f"  All gaps > 0.5: {(top50['opportunity_gap'] > 0.5).all()}")
print(f"  Reason code distribution in top 50:")
for rc in ['under_indexed_decay', 'high_demand_opportunity', 'stale_authority']:
    n = top50['reason_code'].str.contains(rc).sum()
    print(f"    {rc}: {n}")
print(f"\nTop 15 refresh candidates:")
print(top50.head(15).to_string(index=False))

# --- Permutation Importance on test set (for capstone report) ---
print(f"\n{'=' * 80}")
print("PERMUTATION IMPORTANCE (HGBR on held-out test clients)")
print("=" * 80)

df_feat = pd.read_parquet(OUTPUT_DIR / 'prepared_features.parquet')
META_COLS = ['client_id', 'content_id', 'target_log_impressions']
feature_cols = [c for c in df_feat.columns if c not in META_COLS]
X_train_full = df_feat.iloc[payload['train_idx']][feature_cols]
y_train_full = df_feat.iloc[payload['train_idx']]['target_log_impressions']
X_test_full = df_feat.iloc[payload['test_idx']][feature_cols]
y_test_full = df_feat.iloc[payload['test_idx']]['target_log_impressions']

from sklearn.ensemble import HistGradientBoostingRegressor
hgbr_model = HistGradientBoostingRegressor(**payload['hgbr_best_params'])
hgbr_model.fit(X_train_full, y_train_full)

perm_imp = permutation_importance(
    hgbr_model, X_test_full, y_test_full, n_repeats=5, random_state=42, scoring='r2'
)

imp_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm_imp.importances_mean,
    'importance_std': perm_imp.importances_std,
}).sort_values('importance_mean', ascending=False)

print(f"\nTop 15 features by test set permutation importance:")
print(imp_df.head(15).to_string(index=False))

imp_df.to_csv(OUTPUT_DIR / 'permutation_importance.csv', index=False)
print(f"\n[PASS] Saved permutation_importance.csv")

Pages with opportunity_gap > 0.5: 1,996

[PASS] Saved refresh_queue_sample.csv (50 rows)
  All gaps > 0.5: True
  Reason code distribution in top 50:
    under_indexed_decay: 45
    high_demand_opportunity: 1
    stale_authority: 9

Top 15 refresh candidates:
        client_id           content_id  pred_impressions  actual_impressions  opportunity_gap                          reason_code
client_8527a891e2 content_7a8513b2abec        842.892855                   1         6.044878                      stale_authority
client_8527a891e2 content_31c66d071a62       2503.722786                   6         5.880023                  under_indexed_decay
client_8527a891e2 content_ed3d1e0c4c93       1198.813292                   3         5.703627                  under_indexed_decay
client_8527a891e2 content_74ed7b370eb8       1058.196565                   3         5.578972                  under_indexed_decay
client_8527a891e2 content_8c482a64a3df        469.800804                   1         


Top 15 features by test set permutation importance:
                   feature  importance_mean  importance_std
         log_pageviews_90d         0.305865        0.003815
          log_sessions_90d         0.089339        0.001298
log_days_since_last_update         0.070512        0.004531
      log_content_age_days         0.051182        0.006451
            log_word_count         0.049337        0.002179
 engagement_volume_synergy         0.034063        0.001010
             log_users_90d         0.029688        0.000653
           serp_rank_power         0.022748        0.001705
       scroll_per_pageview         0.010785        0.001086
      imputed_avg_position         0.009486        0.000873
     engaged_session_ratio         0.008771        0.000667
        missing_word_count         0.004783        0.000621
               scroll_rate         0.003095        0.000743
       position_tier_top_3         0.002387        0.000524
           engagement_rate         0.002024    